# GEOG 592 — Module 4 Homework, Clean CSV I found
Hailey Rodriguez

Find a CSV file with at least 100 rows—ideally more than 1,000 but fewer than 10,000. It should contain a field that could potentially be used to join the CSV to a vector dataset.

Clean the CSV file using Python’s built-in csv module. 

Place your Jupyter Notebook under homework/week04/

Place your csv and clean csv under
Place your JN under homework/week04/data/

 

Put the link to your GitHub homework/week04/data/ folder in Canvas 


# Module 04 — Finding and Cleaning a CSV

**Dataset:** SCDNR "Species County Distributions" — rare, threatened, and endangered species observed by county in South Carolina.

**Source:** South Carolina Department of Natural Resources (SCDNR), Natural Heritage Program. Pulled directly from their public ArcGIS REST feature service:

`https://arcweb.dnr.sc.gov/server/rest/services/Hosted/Species_County_Distributions/FeatureServer/1`

- **Rows:** 6,743 (one row per species observed in one SC county)
- **Join field:** `county` — matches directly to the `NAME` field on a standard county boundary layer (e.g. Census TIGER/Line counties for South Carolina, or SC county GIS boundaries), so this table can be joined to a vector polygon layer for mapping.
- **Cleaning tool:** Python's built-in `csv` module only (no pandas).

On cleaning my data:
- spp_list was a repeating blob that was duplicating in every row
- Dropped coutynm because it's the same as county
- dropped empty rows (stlength__, created_user, created_date, starea__), plus GIS join-bookkeeping fields (globalid, target_fid, join_fid, join_count, orig_fid, last_edited_user) and geometry stats that mean nothing without the actual geometry (SHAPE__Area, SHAPE__Length).
- Dropped last_editited_date same as timestamp 
- Renamed objectid to record_id
- Removed whitepsace
- Took empty locuseclass values and put "Not Recored" instead 
- All rows remain the same, columns were edited. 

In [4]:
import csv

RAW_PATH = "scdnr_species_county_raw.csv"
CLEAN_PATH = "scdnr_species_county_clean.csv"

with open(RAW_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    raw_fieldnames = reader.fieldnames
    raw_rows = list(reader)

print("raw rows:", len(raw_rows))
print("raw columns:", len(raw_fieldnames))

raw rows: 6743
raw columns: 40


Going Through Raw Data

In [5]:
# county vs countynm:  same field twice?
county_matches = sum(1 for r in raw_rows if r["county"] == r["countynm"])
print(f"county == countynm for all rows? {county_matches == len(raw_rows)} "
      f"({county_matches} / {len(raw_rows)})")

# which columns are empty in every row?
empty_counts = {col: 0 for col in raw_fieldnames}
for row in raw_rows:
    for col, val in row.items():
        if val is None or val.strip() == "":
            empty_counts[col] += 1

always_empty = [c for c, n in empty_counts.items() if n == len(raw_rows)]
print("\nColumns that are empty in every single row:")
for c in always_empty:
    print(" ", c)

mostly_empty = max(
    ((c, n) for c, n in empty_counts.items() if 0 < n < len(raw_rows)),
    key=lambda item: item[1],
)
print("\nColumn with the most missing (but not all) values:")
print(f"  {mostly_empty[0]}: {mostly_empty[1]} / {len(raw_rows)} blank")

led_values = {row["last_edited_date"] for row in raw_rows}
print(f"\nlast_edited_date has {len(led_values)} distinct value across all rows "
      f"(a single bulk-load timestamp, not per-record info): {led_values}")

seen = set()
dup_count = 0
for row in raw_rows:
    key = tuple(v for k, v in row.items() if k != "spp_list")
    if key in seen:
        dup_count += 1
    seen.add(key)
print("\nExact duplicate rows (ignoring spp_list):", dup_count)

county == countynm for all rows? True (6743 / 6743)

Columns that are empty in every single row:
  stlength__
  created_user
  created_date
  starea__

Column with the most missing (but not all) values:
  locuseclass: 2621 / 6743 blank

last_edited_date has 1 distinct value across all rows (a single bulk-load timestamp, not per-record info): {'1714415800000'}

Exact duplicate rows (ignoring spp_list): 0


In [6]:
DROP_COLUMNS = [
    "spp_list", "globalid", "stlength__", "target_fid", "countynm",
    "last_edited_date", "join_count", "join_fid", "SHAPE__Area",
    "created_user", "orig_fid", "last_edited_user", "created_date",
    "starea__", "SHAPE__Length",
]

COLUMN_ORDER = [
    "record_id", "county", "scientific_name", "s_common_name", "inf_taxon",
    "group_member_type_desc", "g_rank", "s_rank", "g1g2s1s2", "state_status",
    "state_protect_status_desc", "fed_status", "fed_protect_desc",
    "listed_species", "tracked_species", "swap_priority_desc",
    "presence_status", "obsrank", "locuseclass", "date_class",
    "first_obs_date", "last_survey_date", "dateconvert", "count_records",
    "eo_id_sc",
]


def clean_row(raw_row):
    row = {k: v for k, v in raw_row.items() if k not in DROP_COLUMNS}
    row["record_id"] = row.pop("objectid")

    for key, value in row.items():
        row[key] = value.strip() if isinstance(value, str) else value

    if row["locuseclass"] == "":
        row["locuseclass"] = "Not Recorded"

    return {col: row[col] for col in COLUMN_ORDER}


cleaned_rows = [clean_row(row) for row in raw_rows]

In [7]:
with open(CLEAN_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMN_ORDER)
    writer.writeheader()
    writer.writerows(cleaned_rows)

print(f"Wrote {len(cleaned_rows)} cleaned rows to {CLEAN_PATH}")

Wrote 6743 cleaned rows to scdnr_species_county_clean.csv


Verification

In [8]:
with open(CLEAN_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    clean_fieldnames = reader.fieldnames
    verify_rows = list(reader)

print("clean rows:", len(verify_rows))
print("clean columns:", len(clean_fieldnames))
print("remaining blank locuseclass values:",
      sum(1 for r in verify_rows if r["locuseclass"] == ""))

print("\nFirst cleaned row:")
for k, v in verify_rows[0].items():
    print(f"  {k}: {v}")

counties = {r["county"] for r in verify_rows}
print("\nDistinct counties represented:", len(counties))

clean rows: 6743
clean columns: 25
remaining blank locuseclass values: 0

First cleaned row:
  record_id: 1
  county: Abbeville
  scientific_name: Alasmidonta varicosa
  s_common_name: Brook Floater
  inf_taxon: Freshwater Mussels
  group_member_type_desc: Zoological
  g_rank: G3
  s_rank: S1S2
  g1g2s1s2: 1
  state_status: Not Applicable
  state_protect_status_desc: Not Applicable
  fed_status: Not Applicable
  fed_protect_desc: Not Applicable
  listed_species: 1
  tracked_species: 1
  swap_priority_desc: Highest
  presence_status: 3
  obsrank: N - Not Ranked
  locuseclass: Not applicable
  date_class: > 40 Years
  first_obs_date: 1960-09-04
  last_survey_date: 1960-09-04
  dateconvert: 1960-09-04
  count_records: 81
  eo_id_sc: 15820

Distinct counties represented: 46


- The `county` field (all 46 South Carolina counties, consistently spelled) is ready to be used as the join key to a county boundary vector layer.

AI Usage

I used ai to helop me find a data source that would be good for this assigment. 